# Create Delta Tables

Reads parquet files from the volume and creates managed Delta tables with comments.

In [ ]:
dbutils.widgets.text("catalog", "startups_catalog")
dbutils.widgets.text("schema", "dw_genie")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"
print(f"Creating tables in {CATALOG}.{SCHEMA}")

In [ ]:
TABLE_COMMENTS = {
    "customers": "Customer accounts with tier (Free/Pro/Enterprise), region, industry segmentation, and geographic coordinates (latitude/longitude).",
    "products": "Product catalog with pricing, cost, and category information.",
    "orders": "Customer orders with status tracking and dates.",
    "order_items": "Individual line items within orders, linking products to orders with quantities and pricing.",
}

COLUMN_COMMENTS = {
    "customers": {
        "customer_id": "Unique customer identifier",
        "name": "Full name of the customer contact",
        "email": "Customer email address",
        "company": "Company or organization name",
        "tier": "Subscription tier: Free, Pro, or Enterprise",
        "region": "Geographic region",
        "industry": "Industry vertical",
        "signup_date": "Date the customer account was created",
        "latitude": "Customer location latitude (WGS84)",
        "longitude": "Customer location longitude (WGS84)",
    },
    "products": {
        "product_id": "Unique product identifier",
        "name": "Product display name",
        "category": "Product category",
        "unit_price": "List price per unit in USD",
        "cost": "Cost of goods sold per unit in USD",
        "is_active": "Whether the product is currently available for purchase",
    },
    "orders": {
        "order_id": "Unique order identifier",
        "customer_id": "FK to customers.customer_id",
        "order_date": "Date the order was placed",
        "status": "Order status: completed, processing, shipped, cancelled, or refunded",
    },
    "order_items": {
        "item_id": "Unique line item identifier",
        "order_id": "FK to orders.order_id",
        "product_id": "FK to products.product_id",
        "quantity": "Number of units ordered",
        "unit_price": "Effective price per unit at time of order (may include discount)",
        "total_price": "Total line item amount (unit_price * quantity)",
    },
}

In [ ]:
for table_name, comment in TABLE_COMMENTS.items():
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    parquet_path = f"{VOLUME_PATH}/{table_name}/"

    # Drop existing table to handle schema changes (e.g. new columns)
    print(f"Dropping {full_name} (if exists)...")
    spark.sql(f"DROP TABLE IF EXISTS {full_name}")

    print(f"Creating {full_name}...")
    df = spark.read.parquet(parquet_path)
    df.write.mode("overwrite").saveAsTable(full_name)

    # Table comment
    spark.sql(f"COMMENT ON TABLE {full_name} IS '{comment}'")

    # Column comments
    for col_name, col_comment in COLUMN_COMMENTS.get(table_name, {}).items():
        spark.sql(f"ALTER TABLE {full_name} ALTER COLUMN {col_name} COMMENT '{col_comment}'")

    row_count = spark.table(full_name).count()
    print(f"  {full_name}: {row_count:,} rows")

print("\nAll tables created!")